## WIDGETS 

In [0]:
dbutils.widgets.text("catalog","ecommerce_catalog_dev")
catalog = dbutils.widgets.get("catalog")

## TABLE CREATION 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog).bronze.raw_orders;

## DATA LOADED INTO TABLE

In [0]:
%sql
COPY INTO IDENTIFIER(:catalog).bronze.raw_orders 
FROM "/Volumes/" :catalog "/bronze/raw_data_volume/ecommerce_transactions/"
FILEFORMAT = CSV 
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'true'
)
COPY_OPTIONS(
    'mergeSchema' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
1050,1050,0


## ADDING NEW COL IN THE EXISTING TABLE

In [0]:
from pyspark.sql.functions import current_timestamp , col

df = spark.table(f"{catalog}.bronze.raw_orders")
df = df.withColumn("ingestion_time" , current_timestamp())\
       .withColumn("source_file" , col("_metadata.file_path"))

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.bronze.raw_orders")
